# Chicago Taxi Trips — week of 11–17 September 2023
SUMMARY_PLACEHOLDER

In [1]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import os
os.makedirs('figures/analysis', exist_ok=True)
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)
BLUE, ORANGE, GREY = '#2a78d6', '#eb6834', '#8a8985'
plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True,
                     'grid.color': '#e6e5e1', 'grid.linewidth': 0.8, 'axes.axisbelow': True,
                     'figure.dpi': 100, 'axes.titlesize': 13, 'axes.titleweight': 'bold'})
raw = pd.read_csv('trips.csv', dtype={'pickup_census_tract': 'string', 'dropoff_census_tract': 'string'},
                  parse_dates=['trip_start_timestamp', 'trip_end_timestamp'])
print(raw.shape)
print(raw.dtypes)

(135768, 23)
trip_id                                  str
taxi_id                                  str
trip_start_timestamp          datetime64[us]
trip_end_timestamp            datetime64[us]
trip_seconds                         float64
trip_miles                           float64
pickup_census_tract                   string
dropoff_census_tract                  string
pickup_community_area                float64
dropoff_community_area               float64
fare                                 float64
tips                                 float64
tolls                                float64
extras                               float64
trip_total                           float64
payment_type                             str
company                                  str
pickup_centroid_latitude             float64
pickup_centroid_longitude            float64
pickup_centroid_location                 str
dropoff_centroid_latitude            float64
dropoff_centroid_longitude           float

In [2]:
print(raw.isna().sum())
print('duplicate trip_id:', raw.trip_id.duplicated().sum())
print(raw.trip_start_timestamp.min(), raw.trip_start_timestamp.max())
print(raw.describe().T)

trip_id                           0
taxi_id                           0
trip_start_timestamp              0
trip_end_timestamp                1
trip_seconds                     33
trip_miles                        0
pickup_census_tract           71404
dropoff_census_tract          73842
pickup_community_area          3815
dropoff_community_area        13794
fare                            324
tips                            324
tolls                           324
extras                          324
trip_total                      324
payment_type                      0
company                           0
pickup_centroid_latitude       3775
pickup_centroid_longitude      3775
pickup_centroid_location       3775
dropoff_centroid_latitude     12872
dropoff_centroid_longitude    12872
dropoff_centroid_location     12872
dtype: int64
duplicate trip_id: 0
2023-09-11 00:00:00 2023-09-17 23:45:00
                               count                        mean                  min             

In [3]:
print(raw.payment_type.value_counts(dropna=False))
print()
print(raw.company.value_counts(dropna=False).to_string())

payment_type
Credit Card    60291
Cash           40053
Mobile         18594
Prcard         11961
Unknown         4784
No Charge         54
Dispute           31
Name: count, dtype: int64

company
Flash Cab                               25950
Taxi Affiliation Services               25727
Sun Taxi                                15122
Taxicab Insurance Agency Llc            13914
City Service                            13747
Chicago Independents                     8004
5 Star Taxi                              6121
Medallion Leasin                         5003
Globe Taxi                               4772
Blue Ribbon Taxi Association             4390
Star North Taxi Management Llc           2815
Taxicab Insurance Agency, LLC            2379
Choice Taxi Association                  2022
Chicago City Taxi Association            1386
Choice Taxi Association Inc              1064
U Taxicab                                1048
Chicago Taxicab                           406
Patriot Taxi Dba Peace 

In [4]:
money = ['fare','tips','tolls','extras','trip_total']
nomoney = raw[money].isna().all(axis=1)
print('rows missing all money fields:', nomoney.sum(), '| any missing:', raw[money].isna().any(axis=1).sum())
print(raw[nomoney][['trip_seconds','trip_miles','payment_type','company']].describe(include='all').T)
print()
# pickup area missing but centroid present
odd = raw.pickup_community_area.isna() & raw.pickup_centroid_latitude.notna()
print('no community area but has centroid:', odd.sum())
print(raw[odd][['pickup_census_tract','pickup_centroid_latitude','pickup_centroid_longitude']].head())
print('trip_end missing:'); print(raw[raw.trip_end_timestamp.isna()].T.head(12))

rows missing all money fields: 324 | any missing: 324
              count unique                           top freq        mean          std  min     25%    50%      75%     max
trip_seconds  316.0    NaN                           NaN  NaN  1231.85443  1166.560076  2.0  398.75  727.0   1923.5  7278.0
trip_miles    324.0    NaN                           NaN  NaN    7.143765     8.608393  0.0  0.8775   2.32  12.9125   34.65
payment_type    324      2                          Cash  323         NaN          NaN  NaN     NaN    NaN      NaN     NaN
company         324     12  Taxicab Insurance Agency Llc  104         NaN          NaN  NaN     NaN    NaN      NaN     NaN

no community area but has centroid: 40
      pickup_census_tract  pickup_centroid_latitude  pickup_centroid_longitude
5285          17031770700                 41.982775                 -87.877305
5912          17031770700                 41.982775                 -87.877305
12033         17031770700                 41.9827

In [5]:
d = raw[~nomoney].copy()
d['parts'] = d[['fare','tips','tolls','extras']].sum(axis=1)
d['diff'] = (d.trip_total - d.parts).round(2)
print('total != sum of parts (>0.01):', (d['diff'].abs() > 0.01).sum())
print(d['diff'].describe())
print(d.loc[d['diff'].abs() > 0.01, ['fare','tips','tolls','extras','trip_total','diff','payment_type']].sort_values('diff').head(8))
print(d.loc[d['diff'].abs() > 0.01, ['fare','tips','tolls','extras','trip_total','diff','payment_type']].sort_values('diff').tail(8))
print('diff value counts (top):'); print(d['diff'].round(2).value_counts().head(8))

total != sum of parts (>0.01): 48197


count    135444.000000
mean          0.204359
std           0.365044
min          -0.000000
25%           0.000000
50%           0.000000
75%           0.500000
max           3.000000
Name: diff, dtype: float64
        fare   tips  tolls  extras  trip_total  diff payment_type
0      25.00   5.00    0.0     0.0        30.5   0.5  Credit Card
89538  45.50  10.60    0.0     7.0        63.6   0.5  Credit Card
89540  31.00   7.10    0.0     4.0        42.6   0.5  Credit Card
89544  25.00   6.00    0.0     4.0        35.5   0.5  Credit Card
89550  58.25  11.75    0.0     0.0        70.5   0.5  Credit Card
89551  44.50   9.80    0.0     4.0        58.8   0.5  Credit Card
89552  33.25   6.95    0.0     1.0        41.7   0.5  Credit Card
89556  10.50   1.50    0.0     0.0        12.5   0.5  Credit Card
         fare   tips  tolls  extras  trip_total  diff payment_type
108354   5.00   3.00    0.0     0.0       11.00   3.0  Credit Card
95826   90.75  18.19    0.0    30.0      141.94   3.0  Credit

In [6]:
print('trip_seconds==0:', (raw.trip_seconds == 0).sum(), '| NaN:', raw.trip_seconds.isna().sum(), '| >6h:', (raw.trip_seconds > 6*3600).sum(), '| >24h:', (raw.trip_seconds >= 86400).sum())
print('trip_miles==0:', (raw.trip_miles == 0).sum(), '| >100:', (raw.trip_miles > 100).sum(), '| >50:', (raw.trip_miles > 50).sum())
print('fare==0 (non-missing):', (d.fare == 0).sum(), '| fare>200:', (d.fare > 200).sum(), '| fare>100:', (d.fare > 100).sum())
print('trip_total==0:', (d.trip_total == 0).sum(), '| >200:', (d.trip_total > 200).sum(), '| tolls>50:', (d.tolls > 50).sum(), '| extras>50:', (d.extras > 50).sum())
print('negative any:', (raw[['trip_seconds','trip_miles']+money] < 0).sum().sum())
print()
print('quantiles'); print(raw[['trip_seconds','trip_miles','fare','trip_total']].quantile([.001,.01,.5,.99,.999,.9999]).round(2))

trip_seconds==0: 2731 | NaN: 33 | >6h: 98 | >24h: 0
trip_miles==0: 13993 | >100: 11 | >50: 75
fare==0 (non-missing): 135 | fare>200: 49 | fare>100: 388
trip_total==0: 134 | >200: 109 | tolls>50: 6 | extras>50: 444
negative any: 0

quantiles


        trip_seconds  trip_miles    fare  trip_total
0.0010          0.00        0.00    0.01        0.01
0.0100          0.00        0.00    3.25        3.25
0.5000       1020.00        3.24   17.75       21.25
0.9900       4872.98       27.14   75.50      104.43
0.9990      12416.09       44.08  150.00      186.53
0.9999      62853.88       83.95  400.00      443.53


In [7]:
cols = ['trip_seconds','trip_miles','fare','tips','tolls','extras','trip_total','payment_type']
print('Top 12 by trip_total'); print(d.nlargest(12,'trip_total')[cols].to_string())
print('\nTop 8 by miles'); print(raw.nlargest(8,'trip_miles')[cols].to_string())
print('\nTop 8 by seconds'); print(raw.nlargest(8,'trip_seconds')[cols].to_string())

Top 12 by trip_total
        trip_seconds  trip_miles    fare    tips    tolls   extras  trip_total payment_type
124235        1080.0        0.10   11.25    0.00  4444.44  4445.44     8901.13         Cash
25636            0.0        0.00    3.25    0.00  4444.44  4444.44     8892.13         Cash
35524          420.0        1.30    6.75    0.00  4111.10     0.44     4118.29         Cash
127990           NaN       37.45  911.00    0.00     0.00     0.00      911.00         Cash
31648            3.0        0.00  900.00    0.00     0.00     0.00      900.00         Cash
99632           60.0        0.00  758.00    0.00     0.00     0.00      758.00    No Charge
30785            0.0        0.00  550.00    0.00     0.00     0.00      550.00  Credit Card
51799            0.0        0.00  550.00    0.00     0.00     0.00      550.00         Cash
124119           0.0        0.00  500.00    0.00     0.00     0.00      500.00         Cash
16606            6.0        0.00  375.00  112.65     0.00  

In [8]:
print('extras distribution'); print(d.extras.describe(percentiles=[.5,.9,.99,.999]).round(2))
print('extras>30:', (d.extras>30).sum(), ' extras>100:', (d.extras>100).sum(), ' tolls>0:', (d.tolls>0).sum())
print(d[d.extras>100][cols].head(10).to_string())
print('\nfare==0 by payment_type:'); print(d[d.fare==0].payment_type.value_counts())
print('\nfare>0 & miles==0:', ((d.fare>0)&(d.trip_miles==0)).sum(), '| fare>0 & seconds==0:', ((d.fare>0)&(d.trip_seconds==0)).sum())
print('miles==0 & seconds==0 & fare>0:', ((d.fare>0)&(d.trip_miles==0)&(d.trip_seconds==0)).sum())
print('\nfare per mile for miles>0 quantiles'); r=(d.fare/d.trip_miles)[d.trip_miles>0]; print(r.quantile([.01,.5,.99,.999]).round(2)); print('fare/mile>50:', (r>50).sum())
print('\nfare>0 by payment_type tips share etc:'); print(d.groupby('payment_type').apply(lambda g: pd.Series({'n':len(g),'diff0.5':(g['diff']==0.5).mean().round(3),'diff>=2.5':(g['diff']>=2.5).mean().round(3)})))

extras distribution
count    135444.00
mean          2.54
std          18.31
min           0.00
50%           0.00
90%           5.00
99%          35.50
99.9%        65.75
max        4445.44
Name: extras, dtype: float64
extras>30: 2017  extras>100: 15  tolls>0: 735
       trip_seconds  trip_miles    fare   tips    tolls   extras  trip_total payment_type
6124           14.0        0.00    3.25  24.75     0.00   120.00      148.50  Credit Card
15069          12.0        0.03    3.25   0.00     0.00   127.00      130.75  Credit Card
25636           0.0        0.00    3.25   0.00  4444.44  4444.44     8892.13         Cash
34658        1680.0        0.20   15.50   0.00     0.00   166.66      182.16         Cash
40849           0.0        0.00    3.25   0.00     0.00   147.00      150.25  Credit Card
63736          17.0        0.00    3.25  31.44     0.00   122.00      157.19  Credit Card
73653         872.0        2.98   12.00   0.00     0.00   104.00      116.50  Credit Card
75080         